# BLIP-2 + PVT v2 + QFormer LoRA — Kaggle Training Notebook

**Architecture:** PVT v2 b2 (frozen) → LayerNorm → QFormer + LoRA (16 queries) → FlanT5-Base  
**Repository:** `archi-dev2/lavis-ai` (`pre-training-stable` branch)  
**Dataset:** MS COCO Karpathy Split  

Features:
- Step-accurate checkpoint resume
- Resume-safe early stopping
- Mixed-precision (fp16) training
- Kaggle timeout safety (periodic saves)
- Gradient accumulation

## 1. Install & Setup

In [ ]:
%%capture
# ── Install core dependencies ──────────────────────────────────────────────
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q transformers==4.33.2 timm==0.4.12 accelerate datasets pillow tqdm
!pip install -q sentencepiece protobuf peft omegaconf iopath
!pip install -q pycocoevalcap pycocotools
!pip install -q fairscale==0.4.4 einops>=0.4.1
!pip install -q kaggle

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. Kaggle API Setup

In [ ]:
import os, shutil, glob

# ── Kaggle API credentials (OPTIONAL) ──────────────────────────────────────
# If you have kaggle.json, upload it as a dataset and attach to this notebook.
# If not, the notebook will use direct HTTP download fallback (no credentials needed).

KAGGLE_JSON_DATASET = "kaggle-api-key"  # Change to your dataset name if using Kaggle API
kaggle_input_path = f"/kaggle/input/{KAGGLE_JSON_DATASET}/kaggle.json"

kaggle_dir = os.path.expanduser("~/.kaggle")
kaggle_json = os.path.join(kaggle_dir, "kaggle.json")

if not os.path.exists(kaggle_json):
    os.makedirs(kaggle_dir, exist_ok=True)
    
    # Try finding kaggle.json in /kaggle/input/
    candidates = glob.glob("/kaggle/input/*/kaggle.json")
    
    if os.path.exists(kaggle_input_path):
        shutil.copy(kaggle_input_path, kaggle_json)
        print("✓ Kaggle API credentials configured.")
    elif candidates:
        shutil.copy(candidates[0], kaggle_json)
        print(f"✓ Found kaggle.json at: {candidates[0]}")
    else:
        print("⚠ Kaggle API credentials NOT found.")
        print("  → No problem! The dataset download will use direct HTTP fallback.")
        print("  → You can still upload kaggle.json later if you prefer Kaggle API.")
else:
    print("✓ Kaggle API credentials already present.")

## 3. Clone Repository

In [ ]:
import os, subprocess, sys

REPO_URL = "https://github.com/archi-dev2/lavis-ai.git"
REPO_BRANCH = "pre-training-stable"
REPO_DIR = "/kaggle/working/lavis-ai"

if not os.path.exists(REPO_DIR):
    print(f"Cloning {REPO_URL} (branch: {REPO_BRANCH})...")
    subprocess.run(
        ["git", "clone", "--depth", "1", "-b", REPO_BRANCH, REPO_URL, REPO_DIR],
        check=True
    )
    print("Clone complete.")
else:
    print(f"Repository already exists at {REPO_DIR}")

# Add repo to sys.path (simpler than pip install -e, avoids dependency conflicts)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# Verify import
try:
    import lavis
    print(f"✓ LAVIS imported from: {lavis.__path__}")
except ImportError as e:
    print(f"⚠ Warning: Could not import LAVIS: {e}")
    print("  Continuing anyway — will try lazy imports during training")

## 4. Imports & Configuration

In [ ]:
import os
import sys
import json
import glob
import time
import random
import logging
from pathlib import Path
from dataclasses import dataclass, field

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
from PIL import Image
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

# ── LAVIS imports (gracefully handle missing dependencies) ────────────────
try:
    from lavis.common.registry import registry
    from lavis.models import load_model
    from lavis.processors import load_processor
except ImportError as e:
    print(f"⚠ Warning: Could not import some LAVIS modules: {e}")
    print("  Continuing anyway — will load model directly\n")

# Try to register modules (optional, for advanced features)
try:
    from lavis.models import *
    from lavis.processors import *
    from lavis.datasets.builders import *
except ImportError:
    pass

# ── Logging ────────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger(__name__)

# ── Reproducibility ────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# ── Training Configuration ─────────────────────────────────────────────────
@dataclass
class Config:
    # Paths
    data_dir: str = "/kaggle/working/coco"
    checkpoint_dir: str = "/kaggle/working/checkpoints"
    repo_dir: str = "/kaggle/working/lavis-ai"
    
    # Model
    vit_model: str = "pvt_v2_b2"
    t5_model: str = "google/flan-t5-base"  # Smaller T5 for faster training (was flan-t5-xl)
    image_size: int = 224
    num_query_token: int = 16
    max_txt_len: int = 32
    prompt: str = "a photo of"
    freeze_vit: bool = True
    
    # Training
    num_epochs: int = 10
    batch_size_train: int = 8       # Optimized for T4 16GB (~20% faster)
    batch_size_eval: int = 8
    gradient_accumulation_steps: int = 4  # Effective batch = 32 (reduced from 8)
    learning_rate: float = 1e-4
    weight_decay: float = 0.05
    warmup_steps: int = 200
    max_grad_norm: float = 1.0
    fp16: bool = True
    
    # Checkpointing
    save_every_n_steps: int = 500   # Save checkpoint every N global steps
    
    # Early Stopping
    patience: int = 3
    min_delta: float = 0.0
    
    # Data
    num_workers: int = 2
    pin_memory: bool = True
    max_train_samples: int = -1     # -1 = use all
    max_val_samples: int = -1       # -1 = use all


cfg = Config()
os.makedirs(cfg.checkpoint_dir, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✓ Device: {DEVICE}")
print(f"✓ Effective batch size: {cfg.batch_size_train * cfg.gradient_accumulation_steps}")

## 5. Dataset Download (MS COCO Karpathy Split)

In [ ]:
import os, subprocess, zipfile

DATA_DIR = cfg.data_dir
os.makedirs(DATA_DIR, exist_ok=True)

# ── Check if data already exists ───────────────────────────────────────────
images_exist = (
    os.path.isdir(os.path.join(DATA_DIR, "images", "train2014"))
    or os.path.isdir(os.path.join(DATA_DIR, "train2014"))
)

# ── Search for pre-attached COCO dataset (recursive search) ────────────────
print("Searching for pre-attached COCO dataset...")
INPUT_COCO_DIRS = []

# COCO data indicators to search for
coco_indicators = [
    "train2014", "val2014", "train2017", "val2017",
    "images", "annotations",
    "train2015", "val2015"
]

# Recursively search /kaggle/input for COCO structure (limit depth to 5)
if os.path.isdir("/kaggle/input"):
    for root, dirs, files in os.walk("/kaggle/input"):
        # Skip if too deep (avoid excessive search)
        depth = root.count(os.sep) - "/kaggle/input".count(os.sep)
        if depth > 5:
            continue
        
        # Check if this directory contains COCO indicators
        has_coco_structure = any(ind in dirs or ind in files for ind in coco_indicators)
        
        if has_coco_structure:
            print(f"\n✓ Found COCO dataset structure at: {root}")
            contents = os.listdir(root)
            print(f"  Contents: {sorted(contents)[:20]}")  # Show first 20 items
            INPUT_COCO_DIRS.append(root)
            # Don't recurse further into this found directory
            dirs.clear()

# Deduplicate
INPUT_COCO_DIRS = list(set(INPUT_COCO_DIRS))

# Print summary
if INPUT_COCO_DIRS:
    print(f"\n✓ Found {len(INPUT_COCO_DIRS)} pre-attached dataset location(s)")
    print("  Will use pre-attached data directly (no download needed).")
    # Set the first found directory as the data source
    if INPUT_COCO_DIRS:
        DATA_DIR = INPUT_COCO_DIRS[0]
        print(f"  Using: {DATA_DIR}")
elif images_exist:
    print("✓ COCO images already present in working directory — skipping download.")
else:
    print("\n⚠ No pre-attached COCO dataset found.")
    print("Available top-level items in /kaggle/input:")
    if os.path.isdir("/kaggle/input"):
        for item in sorted(os.listdir("/kaggle/input"))[:10]:
            item_path = os.path.join("/kaggle/input", item)
            if os.path.isdir(item_path):
                try:
                    subcount = len(os.listdir(item_path))
                    print(f"  {item}/ ({subcount} items)")
                except:
                    print(f"  {item}/ (permission denied)")
    print("\nStarting download...")
    
    # COCO 2017 dataset
    COCO_DATASET = "awsaf49/coco-2017-dataset"
    
    try:
        subprocess.run(
            ["kaggle", "datasets", "download", "-d", COCO_DATASET, "-p", DATA_DIR],
            check=True,
            timeout=300
        )
        
        # Unzip
        for zf in glob.glob(os.path.join(DATA_DIR, "*.zip")):
            print(f"Extracting {zf}...")
            with zipfile.ZipFile(zf, 'r') as z:
                z.extractall(DATA_DIR)
            os.remove(zf)
            
        print("COCO download and extraction complete.")
    except Exception as e:
        print(f"Kaggle API download failed: {e}")
        print("Falling back to direct download...")
        
        os.makedirs(os.path.join(DATA_DIR, "images"), exist_ok=True)
        for split in ["train2014", "val2014"]:
            url = f"http://images.cocodataset.org/zips/{split}.zip"
            dest = os.path.join(DATA_DIR, f"{split}.zip")
            if not os.path.exists(os.path.join(DATA_DIR, "images", split)):
                print(f"Downloading {url}...")
                subprocess.run(["wget", "-q", "--show-progress", url, "-O", dest], check=True)
                with zipfile.ZipFile(dest, 'r') as z:
                    z.extractall(os.path.join(DATA_DIR, "images"))
                os.remove(dest)

# ── Verify structure ───────────────────────────────────────────────────────
print("\nData directory contents:")
for root, dirs, files in os.walk(DATA_DIR):
    level = root.replace(DATA_DIR, "").count(os.sep)
    if level < 2:
        indent = " " * 2 * level
        print(f"{indent}{os.path.basename(root)}/")
        if files:
            print(f"{indent}  ({len(files)} files)")


## 6. Dataset Preparation (Karpathy Split)

In [ ]:
import os, json, urllib.request

DATA_DIR = cfg.data_dir
KARPATHY_JSON = os.path.join(DATA_DIR, "dataset_coco.json")

# ── Download Karpathy split if not present ─────────────────────────────────
if not os.path.exists(KARPATHY_JSON):
    # Check /kaggle/input
    candidates = glob.glob("/kaggle/input/**/dataset_coco.json", recursive=True)
    if candidates:
        KARPATHY_JSON = candidates[0]
        print(f"Found Karpathy JSON at: {KARPATHY_JSON}")
    else:
        print("Downloading Karpathy COCO split JSON...")
        KARPATHY_URL = "https://cs.stanford.edu/people/karpathy/deepimagesent/caption_datasets.zip"
        zip_path = os.path.join(DATA_DIR, "caption_datasets.zip")
        urllib.request.urlretrieve(KARPATHY_URL, zip_path)
        
        import zipfile
        with zipfile.ZipFile(zip_path, 'r') as z:
            z.extractall(DATA_DIR)
        os.remove(zip_path)
        print("Karpathy split downloaded.")
else:
    print(f"Karpathy JSON already exists: {KARPATHY_JSON}")

# ── Load and parse ─────────────────────────────────────────────────────────
with open(KARPATHY_JSON, "r") as f:
    karpathy_data = json.load(f)

print(f"Total images in Karpathy split: {len(karpathy_data['images'])}")

# Count splits
split_counts = {}
for img in karpathy_data["images"]:
    s = img["split"]
    split_counts[s] = split_counts.get(s, 0) + 1
print(f"Splits: {split_counts}")

In [ ]:
# ── Build annotation lists compatible with BLIP-2 ──────────────────────────

def resolve_image_path(filename, data_dir):
        """Find the actual image path across common COCO directory layouts."""
            candidates = [
                    os.path.join(data_dir, "images", filename),
                            os.path.join(data_dir, filename),
                                    os.path.join(data_dir, "train2014", filename),
                                            os.path.join(data_dir, "val2014", filename),
                                                    os.path.join(data_dir, "images", "train2014", filename),
                                                            os.path.join(data_dir, "images", "val2014", filename),
                                                                    os.path.join(data_dir, "train2017", filename),
                                                                            os.path.join(data_dir, "val2017", filename),
                                                                                    os.path.join(data_dir, "images", "train2017", filename),
                                                                                            os.path.join(data_dir, "images", "val2017", filename),
                                                                                                ]
                                                                                                    
                                                                                                        for c in candidates:
                                                                                                                if os.path.exists(c):
                                                                                                                            return c
                                                                                                                                return None


                                                                                                                                def build_annotations(karpathy_data, data_dir, split_name):
                                                                                                                                    """
                                                                                                                                        Build flat annotation list from Karpathy JSON.
                                                                                                                                            Each entry: {"image": abs_path, "caption": str, "image_id": int}
                                                                                                                                                Maps 'restval' → 'train' as is standard.
                                                                                                                                                    """
                                                                                                                                                        target_splits = [split_name]
                                                                                                                                                            if split_name == "train":
                                                                                                                                                                    target_splits.append("restval")
                                                                                                                                                                        
                                                                                                                                                                            annotations = []
                                                                                                                                                                                skipped = 0
                                                                                                                                                                                    
                                                                                                                                                                                        for img_info in karpathy_data["images"]:
                                                                                                                                                                                                if img_info["split"] not in target_splits:
                                                                                                                                                                                                            continue
                                                                                                                                                                                                                    
                                                                                                                                                                                                                            filename = img_info["filename"]
                                                                                                                                                                                                                                    img_path = resolve_image_path(filename, data_dir)
                                                                                                                                                                                                                                            
                                                                                                                                                                                                                                                    if img_path is None:
                                                                                                                                                                                                                                                                skipped += 1
                                                                                                                                                                                                                                                                            continue
                                                                                                                                                                                                                                                                                    
                                                                                                                                                                                                                                                                                            img_id = img_info.get("cocoid", img_info.get("imgid", 0))
                                                                                                                                                                                                                                                                                                    
                                                                                                                                                                                                                                                                                                            for sent in img_info["sentences"]:
                                                                                                                                                                                                                                                                                                                        caption = sent["raw"].strip()
                                                                                                                                                                                                                                                                                                                                    if caption:
                                                                                                                                                                                                                                                                                                                                                    annotations.append({
                                                                                                                                                                                                                                                                                                                                                                        "image": img_path,
                                                                                                                                                                                                                                                                                                                                                                                            "caption": caption,
                                                                                                                                                                                                                                                                                                                                                                                                                "image_id": img_id,
                                                                                                                                                                                                                                                                                                                                                                                                                                })
                                                                                                                                                                                                                                                                                                                                                                                                                                    
                                                                                                                                                                                                                                                                                                                                                                                                                                        if skipped > 0:
                                                                                                                                                                                                                                                                                                                                                                                                                                                print(f"  [{split_name}] Skipped {skipped} images (not found on disk)")
                                                                                                                                                                                                                                                                                                                                                                                                                                                    
                                                                                                                                                                                                                                                                                                                                                                                                                                                        return annotations


                                                                                                                                                                                                                                                                                                                                                                                                                                                        train_annotations = build_annotations(karpathy_data, DATA_DIR, "train")
                                                                                                                                                                                                                                                                                                                                                                                                                                                        val_annotations = build_annotations(karpathy_data, DATA_DIR, "val")
                                                                                                                                                                                                                                                                                                                                                                                                                                                        test_annotations = build_annotations(karpathy_data, DATA_DIR, "test")

                                                                                                                                                                                                                                                                                                                                                                                                                                                        print(f"Train annotations: {len(train_annotations)}")
                                                                                                                                                                                                                                                                                                                                                                                                                                                        print(f"Val annotations:   {len(val_annotations)}")
                                                                                                                                                                                                                                                                                                                                                                                                                                                        print(f"Test annotations:  {len(test_annotations)}")

                                                                                                                                                                                                                                                                                                                                                                                                                                                        # Apply sample limits if set
                                                                                                                                                                                                                                                                                                                                                                                                                                                        if cfg.max_train_samples > 0:
                                                                                                                                                                                                                                                                                                                                                                                                                                                            train_annotations = train_annotations[:cfg.max_train_samples]
                                                                                                                                                                                                                                                                                                                                                                                                                                                                print(f"Limiting train to {len(train_annotations)} samples")
                                                                                                                                                                                                                                                                                                                                                                                                                                                                if cfg.max_val_samples > 0:
                                                                                                                                                                                                                                                                                                                                                                                                                                                                    val_annotations = val_annotations[:cfg.max_val_samples]
                                                                                                                                                                                                                                                                                                                                                                                                                                                                        print(f"Limiting val to {len(val_annotations)} samples")
                                                                                                                                                                                                                                                                                                                                                                                                                                                                        

In [ ]:
# ── PyTorch Dataset ────────────────────────────────────────────────────────

from torchvision import transforms


class COCOCaptionDataset(Dataset):
    """COCO Caption dataset for BLIP-2 training."""
    
    def __init__(self, annotations, image_size=224, is_train=True):
        self.annotations = annotations
        
        # Preprocessing consistent with BLIP-2
        if is_train:
            self.transform = transforms.Compose([
                transforms.RandomResizedCrop(
                    image_size, scale=(0.5, 1.0),
                    interpolation=transforms.InterpolationMode.BICUBIC,
                ),
                transforms.RandomHorizontalFlip(),
                transforms.ToTensor(),
                transforms.Normalize(
                    mean=[0.485, 0.456, 0.406],
                    std=[0.229, 0.224, 0.225],
                ),
            ])
        else:
            self.transform = transforms.Compose([
                transforms.Resize(
                    (image_size, image_size),
                    interpolation=transforms.InterpolationMode.BICUBIC,
                ),
                transforms.ToTensor(),
                transforms.Normalize(
                    mean=[0.485, 0.456, 0.406],
                    std=[0.229, 0.224, 0.225],
                ),
            ])
    
    def __len__(self):
        return len(self.annotations)
    
    def __getitem__(self, idx):
        """Return dict compatible with Blip2T5.forward()."""
        max_retries = 5
        for attempt in range(max_retries):
            try:
                ann = self.annotations[idx]
                image = Image.open(ann["image"]).convert("RGB")
                image = self.transform(image)
                caption = ann["caption"]
                
                return {
                    "image": image,
                    "text_input": caption,   # Used as prompt context
                    "text_output": caption,   # Target for loss computation
                    "image_id": ann["image_id"],
                }
            except Exception as e:
                if attempt == max_retries - 1:
                    raise RuntimeError(f"Failed to load sample {idx} after {max_retries} retries: {e}")
                idx = random.randint(0, len(self.annotations) - 1)


def collate_fn(batch):
    """Custom collate: stack images, keep text as lists."""
    images = torch.stack([b["image"] for b in batch])
    return {
        "image": images,
        "text_input": [b["text_input"] for b in batch],
        "text_output": [b["text_output"] for b in batch],
        "image_id": [b["image_id"] for b in batch],
    }


# ── Create datasets and dataloaders ────────────────────────────────────────
train_dataset = COCOCaptionDataset(train_annotations, cfg.image_size, is_train=True)
val_dataset = COCOCaptionDataset(val_annotations, cfg.image_size, is_train=False)

train_loader = DataLoader(
    train_dataset,
    batch_size=cfg.batch_size_train,
    shuffle=True,
    num_workers=cfg.num_workers,
    pin_memory=cfg.pin_memory,
    collate_fn=collate_fn,
    drop_last=True,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=cfg.batch_size_eval,
    shuffle=False,
    num_workers=cfg.num_workers,
    pin_memory=cfg.pin_memory,
    collate_fn=collate_fn,
)

print(f"Train: {len(train_dataset)} samples, {len(train_loader)} batches")
print(f"Val:   {len(val_dataset)} samples, {len(val_loader)} batches")

## 7. Model Initialization

In [ ]:
from lavis.models.blip2_models.blip2_t5 import Blip2T5

# ── Initialize BLIP-2 (PVT v2 + QFormer + FlanT5) ─────────────────────────
print("Initializing BLIP-2 model...")
print(f"  Vision encoder : {cfg.vit_model}")
print(f"  Query tokens   : {cfg.num_query_token}")
print(f"  LLM            : {cfg.t5_model}")
print(f"  Image size     : {cfg.image_size}")

model = Blip2T5(
    vit_model=cfg.vit_model,
    img_size=cfg.image_size,
    drop_path_rate=0,
    use_grad_checkpoint=True,  # Enable for speed/memory tradeoff
    vit_precision="fp16",
    freeze_vit=cfg.freeze_vit,
    num_query_token=cfg.num_query_token,
    t5_model=cfg.t5_model,
    prompt=cfg.prompt,
    max_txt_len=cfg.max_txt_len,
)

model = model.to(DEVICE)

# ── Parameter summary ──────────────────────────────────────────────────────
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\nParameter Summary:")
print(f"  Total:     {total_params:>15,}")
print(f"  Trainable: {trainable_params:>15,}")
print(f"  Frozen:    {total_params - trainable_params:>15,}")
print(f"  Trainable: {100 * trainable_params / total_params:.2f}%")

# Show trainable module breakdown
print(f"\nTrainable modules:")
trainable_by_module = {}
for name, p in model.named_parameters():
    if p.requires_grad:
        module = name.split(".")[0]
        trainable_by_module[module] = trainable_by_module.get(module, 0) + p.numel()
for mod, count in sorted(trainable_by_module.items(), key=lambda x: -x[1]):
    print(f"  {mod:20s}: {count:>12,} ({100*count/trainable_params:.1f}%)")

## 8. Training Functions

In [ ]:
def train_one_epoch(
    model,
    train_loader,
    optimizer,
    scheduler,
    scaler,
    epoch,
    global_step,
    cfg,
    device,
    resume_step_in_epoch=0,
):
    """
    Train one epoch with step-accurate resume support.
    
    Args:
        resume_step_in_epoch: Number of batches already processed this epoch
                              (for mid-epoch resume). 0 = start from beginning.
    
    Returns:
        avg_loss, global_step
    """
    model.train()
    total_loss = 0.0
    num_batches = 0
    optimizer.zero_grad()
    
    pbar = tqdm(
        enumerate(train_loader),
        total=len(train_loader),
        desc=f"Epoch {epoch}",
        dynamic_ncols=True,
    )
    
    for batch_idx, samples in pbar:
        # ── Skip already-processed batches on resume ──────────────────────
        if batch_idx < resume_step_in_epoch:
            if batch_idx % 100 == 0:
                pbar.set_postfix_str(f"skipping to step {resume_step_in_epoch}")
            continue
        
        # Move images to device
        samples["image"] = samples["image"].to(device, non_blocking=True)
        
        # ── Forward pass (mixed precision) ────────────────────────────────
        with autocast(enabled=cfg.fp16):
            output = model(samples)
            loss = output["loss"] / cfg.gradient_accumulation_steps
        
        # ── Backward pass ─────────────────────────────────────────────────
        scaler.scale(loss).backward()
        
        # ── Optimizer step (every N accumulation steps) ───────────────────
        if (batch_idx + 1) % cfg.gradient_accumulation_steps == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.max_grad_norm)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            scheduler.step()
            global_step += 1
            
            # ── Periodic checkpoint (Kaggle timeout safety) ───────────────
            if cfg.save_every_n_steps > 0 and global_step % cfg.save_every_n_steps == 0:
                save_checkpoint(
                    model, optimizer, scheduler, scaler,
                    epoch=epoch,
                    global_step=global_step,
                    batch_idx_in_epoch=batch_idx + 1,
                    best_val_loss=float("inf"),  # Will be updated at epoch end
                    early_stopping_counter=0,
                    checkpoint_dir=cfg.checkpoint_dir,
                    tag="periodic",
                )
        
        # ── Logging ───────────────────────────────────────────────────────
        batch_loss = loss.item() * cfg.gradient_accumulation_steps
        total_loss += batch_loss
        num_batches += 1
        
        pbar.set_postfix({
            "loss": f"{batch_loss:.4f}",
            "avg": f"{total_loss / num_batches:.4f}",
            "lr": f"{scheduler.get_last_lr()[0]:.2e}",
            "step": global_step,
        })
    
    avg_loss = total_loss / max(num_batches, 1)
    return avg_loss, global_step


@torch.no_grad()
def validate(model, val_loader, cfg, device):
    """Run validation and return average loss."""
    model.eval()
    total_loss = 0.0
    num_batches = 0
    
    pbar = tqdm(val_loader, desc="Validation", dynamic_ncols=True)
    
    for samples in pbar:
        samples["image"] = samples["image"].to(device, non_blocking=True)
        
        with autocast(enabled=cfg.fp16):
            output = model(samples)
            loss = output["loss"]
        
        total_loss += loss.item()
        num_batches += 1
        pbar.set_postfix({"val_loss": f"{total_loss / num_batches:.4f}"})
    
    avg_loss = total_loss / max(num_batches, 1)
    return avg_loss

## 9. Checkpoint Save / Load Logic

**Saves everything needed for exact resume:**
- Model, optimizer, scheduler, scaler state dicts
- Epoch, global step, batch index within epoch
- Best validation loss and early stopping counter

In [ ]:
def save_checkpoint(
    model,
    optimizer,
    scheduler,
    scaler,
    epoch,
    global_step,
    batch_idx_in_epoch,
    best_val_loss,
    early_stopping_counter,
    checkpoint_dir,
    tag="epoch",
):
    """
    Save a full training checkpoint for exact resume.
    
    Saves:
        - model.state_dict
        - optimizer.state_dict
        - scheduler.state_dict
        - scaler.state_dict
        - epoch, global_step, batch_idx_in_epoch
        - best_val_loss, early_stopping_counter
    """
    os.makedirs(checkpoint_dir, exist_ok=True)
    
    checkpoint = {
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(),
        "scaler": scaler.state_dict(),
        "epoch": epoch,
        "global_step": global_step,
        "batch_idx_in_epoch": batch_idx_in_epoch,
        "best_val_loss": best_val_loss,
        "early_stopping_counter": early_stopping_counter,
        "config": {
            "learning_rate": cfg.learning_rate,
            "batch_size": cfg.batch_size_train,
            "gradient_accumulation_steps": cfg.gradient_accumulation_steps,
            "image_size": cfg.image_size,
            "num_query_token": cfg.num_query_token,
        },
    }
    
    # Save with tag
    path = os.path.join(checkpoint_dir, f"checkpoint_{tag}_e{epoch}_s{global_step}.pt")
    torch.save(checkpoint, path)
    
    # Also save as 'latest' for easy resume
    latest_path = os.path.join(checkpoint_dir, "checkpoint_latest.pt")
    torch.save(checkpoint, latest_path)
    
    # Save best model separately
    if tag == "best":
        best_path = os.path.join(checkpoint_dir, "checkpoint_best.pt")
        torch.save(checkpoint, best_path)
    
    size_mb = os.path.getsize(path) / 1e6
    logger.info(
        f"Checkpoint saved: {os.path.basename(path)} "
        f"({size_mb:.1f} MB) | epoch={epoch}, step={global_step}, "
        f"best_val_loss={best_val_loss:.4f}, es_counter={early_stopping_counter}"
    )
    
    # ── Cleanup old periodic checkpoints (keep last 3) ────────────────────
    if tag == "periodic":
        periodic_files = sorted(
            glob.glob(os.path.join(checkpoint_dir, "checkpoint_periodic_*.pt")),
            key=os.path.getmtime,
        )
        while len(periodic_files) > 3:
            os.remove(periodic_files.pop(0))
    
    return path


def load_checkpoint(model, optimizer, scheduler, scaler, checkpoint_dir, device, tag=None):
    """
    Load checkpoint and restore ALL training state.
    
    Returns:
        dict with: epoch, global_step, batch_idx_in_epoch,
                   best_val_loss, early_stopping_counter
        or None if no checkpoint found.
    """
    if tag:
        path = os.path.join(checkpoint_dir, f"checkpoint_{tag}.pt")
    else:
        path = os.path.join(checkpoint_dir, "checkpoint_latest.pt")
    
    if not os.path.exists(path):
        logger.info("No checkpoint found. Starting from scratch.")
        return None
    
    logger.info(f"Loading checkpoint: {path}")
    checkpoint = torch.load(path, map_location=device)
    
    # Restore model
    model.load_state_dict(checkpoint["model"], strict=False)
    
    # Restore optimizer
    optimizer.load_state_dict(checkpoint["optimizer"])
    
    # Restore scheduler
    scheduler.load_state_dict(checkpoint["scheduler"])
    
    # Restore scaler
    scaler.load_state_dict(checkpoint["scaler"])
    
    state = {
        "epoch": checkpoint["epoch"],
        "global_step": checkpoint["global_step"],
        "batch_idx_in_epoch": checkpoint.get("batch_idx_in_epoch", 0),
        "best_val_loss": checkpoint["best_val_loss"],
        "early_stopping_counter": checkpoint["early_stopping_counter"],
    }
    
    logger.info(
        f"Resumed from: epoch={state['epoch']}, step={state['global_step']}, "
        f"batch_in_epoch={state['batch_idx_in_epoch']}, "
        f"best_val_loss={state['best_val_loss']:.4f}, "
        f"es_counter={state['early_stopping_counter']}"
    )
    
    return state


def load_latest_checkpoint(model, optimizer, scheduler, scaler, checkpoint_dir, device):
    """Convenience wrapper: auto-detect and load the latest checkpoint."""
    return load_checkpoint(model, optimizer, scheduler, scaler, checkpoint_dir, device, tag=None)

## 10. Training Loop

Full training loop with:
- Auto-resume from latest checkpoint
- Step-accurate mid-epoch resume
- Early stopping (resume-safe)
- Periodic and end-of-epoch checkpointing

In [ ]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts, LambdaLR
import math


def get_cosine_schedule_with_warmup(optimizer, num_warmup_steps, num_training_steps):
    """Cosine LR schedule with linear warmup."""
    def lr_lambda(current_step):
        if current_step < num_warmup_steps:
            return float(current_step) / float(max(1, num_warmup_steps))
        progress = float(current_step - num_warmup_steps) / float(
            max(1, num_training_steps - num_warmup_steps)
        )
        return max(0.0, 0.5 * (1.0 + math.cos(math.pi * progress)))
    return LambdaLR(optimizer, lr_lambda)


# ── Optimizer (only trainable parameters) ─────────────────────────────────
trainable_params = [p for p in model.parameters() if p.requires_grad]
optimizer = AdamW(trainable_params, lr=cfg.learning_rate, weight_decay=cfg.weight_decay)

# ── Scheduler ──────────────────────────────────────────────────────────────
steps_per_epoch = len(train_loader) // cfg.gradient_accumulation_steps
total_training_steps = steps_per_epoch * cfg.num_epochs

scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=cfg.warmup_steps,
    num_training_steps=total_training_steps,
)

# ── Mixed precision scaler ────────────────────────────────────────────────
scaler = GradScaler(enabled=cfg.fp16)

print(f"Steps per epoch: {steps_per_epoch}")
print(f"Total training steps: {total_training_steps}")
print(f"Warmup steps: {cfg.warmup_steps}")

# ══════════════════════════════════════════════════════════════════════════
# AUTO-RESUME: Load latest checkpoint if available
# ══════════════════════════════════════════════════════════════════════════
start_epoch = 0
global_step = 0
resume_batch_idx = 0
best_val_loss = float("inf")
early_stopping_counter = 0

resume_state = load_latest_checkpoint(
    model, optimizer, scheduler, scaler, cfg.checkpoint_dir, DEVICE
)

if resume_state is not None:
    start_epoch = resume_state["epoch"]
    global_step = resume_state["global_step"]
    resume_batch_idx = resume_state["batch_idx_in_epoch"]
    best_val_loss = resume_state["best_val_loss"]
    early_stopping_counter = resume_state["early_stopping_counter"]
    
    # If we saved at the end of an epoch (batch_idx == 0 or full epoch done),
    # move to next epoch
    if resume_batch_idx == 0 or resume_batch_idx >= len(train_loader):
        start_epoch += 1
        resume_batch_idx = 0
    
    print(f"\n{'='*60}")
    print(f"RESUMING TRAINING")
    print(f"  Epoch:           {start_epoch} (batch {resume_batch_idx})")
    print(f"  Global step:     {global_step}")
    print(f"  Best val loss:   {best_val_loss:.4f}")
    print(f"  ES counter:      {early_stopping_counter}/{cfg.patience}")
    print(f"{'='*60}\n")


# ══════════════════════════════════════════════════════════════════════════
# TRAINING LOOP
# ══════════════════════════════════════════════════════════════════════════
train_losses = []
val_losses = []

for epoch in range(start_epoch, cfg.num_epochs):
    print(f"\n{'━'*60}")
    print(f"Epoch {epoch}/{cfg.num_epochs - 1}")
    print(f"{'━'*60}")
    
    # ── 1. Train ──────────────────────────────────────────────────────────
    # On the first epoch after resume, skip already-processed batches
    skip_batches = resume_batch_idx if epoch == start_epoch else 0
    
    train_loss, global_step = train_one_epoch(
        model, train_loader, optimizer, scheduler, scaler,
        epoch=epoch,
        global_step=global_step,
        cfg=cfg,
        device=DEVICE,
        resume_step_in_epoch=skip_batches,
    )
    train_losses.append(train_loss)
    
    # ── 2. Validate ───────────────────────────────────────────────────────
    val_loss = validate(model, val_loader, cfg, DEVICE)
    val_losses.append(val_loss)
    
    print(f"\nEpoch {epoch} Summary:")
    print(f"  Train Loss: {train_loss:.4f}")
    print(f"  Val Loss:   {val_loss:.4f}")
    print(f"  Best Val:   {best_val_loss:.4f}")
    
    # ── 3. Early Stopping Logic (resume-safe) ─────────────────────────────
    if val_loss < best_val_loss - cfg.min_delta:
        # Improvement!
        best_val_loss = val_loss
        early_stopping_counter = 0
        print(f"  ★ New best validation loss! Counter reset.")
        
        # Save best model
        save_checkpoint(
            model, optimizer, scheduler, scaler,
            epoch=epoch,
            global_step=global_step,
            batch_idx_in_epoch=0,  # Full epoch complete
            best_val_loss=best_val_loss,
            early_stopping_counter=early_stopping_counter,
            checkpoint_dir=cfg.checkpoint_dir,
            tag="best",
        )
    else:
        early_stopping_counter += 1
        print(f"  ⚠ No improvement. ES counter: {early_stopping_counter}/{cfg.patience}")
    
    # ── 4. End-of-epoch checkpoint ────────────────────────────────────────
    save_checkpoint(
        model, optimizer, scheduler, scaler,
        epoch=epoch,
        global_step=global_step,
        batch_idx_in_epoch=0,  # Full epoch complete
        best_val_loss=best_val_loss,
        early_stopping_counter=early_stopping_counter,
        checkpoint_dir=cfg.checkpoint_dir,
        tag="epoch",
    )
    
    # ── 5. Check early stopping ───────────────────────────────────────────
    if early_stopping_counter >= cfg.patience:
        print(f"\n{'='*60}")
        print(f"EARLY STOPPING triggered at epoch {epoch}")
        print(f"Best validation loss: {best_val_loss:.4f}")
        print(f"{'='*60}")
        break

print(f"\n{'═'*60}")
print(f"Training Complete!")
print(f"  Final epoch:     {epoch}")
print(f"  Global steps:    {global_step}")
print(f"  Best val loss:   {best_val_loss:.4f}")
print(f"{'═'*60}")

## 11. Resume Training (Standalone Runnable)

Run this cell independently to resume training from the latest checkpoint.  
All state (model, optimizer, scheduler, early stopping) is restored exactly.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# STANDALONE RESUME CELL
# Run this cell to continue training from the last checkpoint.
# Prerequisites: Cells 1-9 must have been run (model, data, functions loaded).
# ══════════════════════════════════════════════════════════════════════════

# Re-create optimizer, scheduler, scaler (fresh, will be overwritten by checkpoint)
_trainable_params = [p for p in model.parameters() if p.requires_grad]
_optimizer = AdamW(_trainable_params, lr=cfg.learning_rate, weight_decay=cfg.weight_decay)
_scheduler = get_cosine_schedule_with_warmup(
    _optimizer,
    num_warmup_steps=cfg.warmup_steps,
    num_training_steps=total_training_steps,
)
_scaler = GradScaler(enabled=cfg.fp16)

# Load checkpoint — restores model, optimizer, scheduler, scaler, and all counters
_state = load_latest_checkpoint(
    model, _optimizer, _scheduler, _scaler, cfg.checkpoint_dir, DEVICE
)

if _state is None:
    print("No checkpoint found. Run the training loop first or check checkpoint_dir.")
else:
    _start_epoch = _state["epoch"]
    _global_step = _state["global_step"]
    _resume_batch = _state["batch_idx_in_epoch"]
    _best_val_loss = _state["best_val_loss"]
    _es_counter = _state["early_stopping_counter"]
    
    # Advance epoch if full epoch was already complete
    if _resume_batch == 0 or _resume_batch >= len(train_loader):
        _start_epoch += 1
        _resume_batch = 0
    
    print(f"Resuming from epoch {_start_epoch}, step {_global_step}, "
          f"batch {_resume_batch}, best_val={_best_val_loss:.4f}, "
          f"es={_es_counter}/{cfg.patience}")
    
    # Continue training
    for _epoch in range(_start_epoch, cfg.num_epochs):
        _skip = _resume_batch if _epoch == _start_epoch else 0
        
        _train_loss, _global_step = train_one_epoch(
            model, train_loader, _optimizer, _scheduler, _scaler,
            epoch=_epoch, global_step=_global_step,
            cfg=cfg, device=DEVICE, resume_step_in_epoch=_skip,
        )
        
        _val_loss = validate(model, val_loader, cfg, DEVICE)
        
        print(f"Epoch {_epoch}: train={_train_loss:.4f}, val={_val_loss:.4f}, "
              f"best={_best_val_loss:.4f}")
        
        # Early stopping
        if _val_loss < _best_val_loss - cfg.min_delta:
            _best_val_loss = _val_loss
            _es_counter = 0
            save_checkpoint(
                model, _optimizer, _scheduler, _scaler,
                epoch=_epoch, global_step=_global_step,
                batch_idx_in_epoch=0, best_val_loss=_best_val_loss,
                early_stopping_counter=_es_counter,
                checkpoint_dir=cfg.checkpoint_dir, tag="best",
            )
        else:
            _es_counter += 1
        
        # End-of-epoch save
        save_checkpoint(
            model, _optimizer, _scheduler, _scaler,
            epoch=_epoch, global_step=_global_step,
            batch_idx_in_epoch=0, best_val_loss=_best_val_loss,
            early_stopping_counter=_es_counter,
            checkpoint_dir=cfg.checkpoint_dir, tag="epoch",
        )
        
        if _es_counter >= cfg.patience:
            print(f"Early stopping at epoch {_epoch}. Best val loss: {_best_val_loss:.4f}")
            break
    
    print("Resume training complete.")

## 12. Inference

In [ ]:
@torch.no_grad()
def generate_captions(model, images, device, num_beams=5, max_length=30):
    """
    Generate captions for a batch of image tensors.
    
    Args:
        model: Blip2T5 model
        images: Tensor of shape (B, 3, H, W)
        device: torch device
    
    Returns:
        List of caption strings
    """
    model.eval()
    images = images.to(device)
    
    samples = {
        "image": images,
        "prompt": cfg.prompt,
    }
    
    captions = model.generate(
        samples,
        use_nucleus_sampling=False,
        num_beams=num_beams,
        max_length=max_length,
        min_length=1,
    )
    
    return captions


# ── Load best checkpoint for inference ────────────────────────────────────
best_ckpt_path = os.path.join(cfg.checkpoint_dir, "checkpoint_best.pt")
latest_ckpt_path = os.path.join(cfg.checkpoint_dir, "checkpoint_latest.pt")

inf_ckpt_path = best_ckpt_path if os.path.exists(best_ckpt_path) else latest_ckpt_path

if os.path.exists(inf_ckpt_path):
    inf_ckpt = torch.load(inf_ckpt_path, map_location=DEVICE)
    model.load_state_dict(inf_ckpt["model"], strict=False)
    print(f"Loaded checkpoint for inference: {os.path.basename(inf_ckpt_path)}")
    print(f"  Epoch: {inf_ckpt['epoch']}, Step: {inf_ckpt['global_step']}, "
          f"Val Loss: {inf_ckpt['best_val_loss']:.4f}")
else:
    print("No checkpoint found — using current model weights.")

# ── Run inference on validation samples ───────────────────────────────────
NUM_SAMPLES = 8

# Get random samples from validation set
sample_indices = random.sample(range(len(val_dataset)), min(NUM_SAMPLES, len(val_dataset)))
sample_batch = collate_fn([val_dataset[i] for i in sample_indices])

# Generate captions
predicted_captions = generate_captions(model, sample_batch["image"], DEVICE)

# Display results
print(f"\n{'='*60}")
print("INFERENCE RESULTS")
print(f"{'='*60}")
for i in range(len(predicted_captions)):
    print(f"\nSample {i+1}:")
    print(f"  Ground Truth: {sample_batch['text_output'][i]}")
    print(f"  Predicted:    {predicted_captions[i]}")

## 13. Visualization

In [ ]:
import matplotlib.pyplot as plt
import textwrap

# ── Unnormalize for display ────────────────────────────────────────────────
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)


def unnormalize(img_tensor):
    """Reverse ImageNet normalization for display."""
    img = img_tensor.cpu() * STD + MEAN
    return img.clamp(0, 1).permute(1, 2, 0).numpy()


# ── Plot predictions ───────────────────────────────────────────────────────
n_show = min(8, len(predicted_captions))
cols = 4
rows = (n_show + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 5 * rows))
axes = axes.flatten() if n_show > 1 else [axes]

for i in range(n_show):
    ax = axes[i]
    img = unnormalize(sample_batch["image"][i])
    ax.imshow(img)
    ax.axis("off")
    
    gt = textwrap.fill(sample_batch["text_output"][i], width=40)
    pred = textwrap.fill(predicted_captions[i], width=40)
    
    ax.set_title(
        f"GT: {gt}\n\nPred: {pred}",
        fontsize=9,
        ha="center",
        va="top",
        wrap=True,
    )

# Hide unused axes
for i in range(n_show, len(axes)):
    axes[i].axis("off")

plt.tight_layout()
plt.savefig("/kaggle/working/predictions.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: /kaggle/working/predictions.png")

In [ ]:
# ── Training curves ────────────────────────────────────────────────────────
if train_losses and val_losses:
    fig, ax = plt.subplots(1, 1, figsize=(10, 5))
    epochs_range = range(len(train_losses))
    
    ax.plot(epochs_range, train_losses, "b-o", label="Train Loss", markersize=5)
    ax.plot(epochs_range, val_losses, "r-s", label="Val Loss", markersize=5)
    ax.axhline(y=best_val_loss, color="green", linestyle="--", alpha=0.5,
               label=f"Best Val: {best_val_loss:.4f}")
    
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.set_title("BLIP-2 (PVT v2 + QFormer LoRA) Training Curves")
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig("/kaggle/working/training_curves.png", dpi=150)
    plt.show()
    print("Saved: /kaggle/working/training_curves.png")
else:
    print("No training history to plot.")

In [ ]:
# ── Final summary ──────────────────────────────────────────────────────────
print("\n" + "═" * 60)
print("NOTEBOOK COMPLETE")
print("═" * 60)
print(f"\nCheckpoints saved to: {cfg.checkpoint_dir}/")

ckpt_files = glob.glob(os.path.join(cfg.checkpoint_dir, "*.pt"))
if ckpt_files:
    print(f"  Files:")
    for f in sorted(ckpt_files):
        size_mb = os.path.getsize(f) / 1e6
        print(f"    {os.path.basename(f):45s} ({size_mb:.1f} MB)")

print(f"\nTo resume: Re-run cells 1–9, then run the 'Resume Training' cell.")
print(f"To infer:  Re-run cells 1–9, then run the 'Inference' cell.")
print("═" * 60)